In [7]:
import pandas as pd

df = pd.read_csv("final_merge.csv")
value_count = df['label_id'].value_counts(normalize=True)   * 100

print(value_count)

label_id
0    39.835390
2    34.730308
1    25.434303
Name: proportion, dtype: float64


In [2]:
%pip install pandas pyvi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import re
import unicodedata
from pyvi import ViTokenizer

# --- DICTIONARIES ---
# Bạn có thể mở rộng danh sách này hoặc đọc từ file .txt
teencode_dict = {
    "đc": "được", "ko": "không", "k": "không", "kh": "không",
    "vcl": "vô cùng luôn", "vkl": "vô cùng luôn", "dcm": "địt con mẹ",
    "đm": "địt mẹ", "cc": "con cặc", "th": "thế", "n": "nó", "m": "mày",
    "t": "tao", "j": "gì", "bt": "bình thường", "v": "vậy"
}

# Danh sách stopword cơ bản (nên dùng file vietnamese-stopwords.txt chuẩn)
stopwords = ["và", "của", "là", "có", "trong", "đã", "cho", "rằng"] 

# --- FUNCTIONS ---

def normalize_unicode(text):
    """Chuẩn hóa Unicode về dạng NFC (UTF-8)"""
    return unicodedata.normalize('NFC', text)

def remove_links(text):
    """Xóa các đường link http/https/www"""
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def remove_redundant_spaces(text):
    """Xóa khoảng trắng thừa"""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def remove_redundant_chars(text):
    """Xóa ký tự lặp lại liên tiếp (ví dụ: haaaaa -> ha)"""
    return re.sub(r'(.)\1{2,}', r'\1', text)

def normalize_accents(text):
    """
    Chuẩn hóa đặt dấu tiếng Việt (Dựa trên quy tắc bạn cung cấp)
    Lưu ý: Thư viện 'underthesea' hoặc 'pyvi' có tích hợp sẵn, 
    nhưng đây là hàm logic mô phỏng quy tắc bạn yêu cầu.
    """
    # Trong thực tế, nên dùng thư viện chuyên dụng để xử lý các case phức tạp:
    # return underthesea.text_normalize(text)
    return text # Tạm thời giữ nguyên vì đa số bộ gõ hiện nay đã chuẩn hóa NFC

def deteen_code(text, dictionary):
    """Chuyển đổi teencode sang từ gốc"""
    words = text.split()
    converted_words = [dictionary.get(w, w) for w in words]
    return " ".join(converted_words)

def preprocess_pipeline(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Xóa Links
    text = remove_links(text)
    
    # 3. Chuẩn hóa Unicode & Dấu
    text = normalize_unicode(text)
    text = normalize_accents(text)
    
    # 4. Xóa ký tự lặp (ví dụ: "vclll" -> "vcl")
    text = remove_redundant_chars(text)
    
    # 5. Xóa khoảng trắng thừa
    text = remove_redundant_spaces(text)
    
    # 6. Word Tokenize (Dùng ViTokenizer của PyVi)
    # Kết quả: "viện trưởng" -> "viện_trưởng"
    text = ViTokenizer.tokenize(text)
    
    # 7. De-teencode
    text = deteen_code(text, teencode_dict)
    
    # 8. Remove Stopwords
    words = text.split()
    words = [w for w in words if w.replace("_", " ") not in stopwords]
    
    return " ".join(words)

# --- MAIN EXECUTION ---

def process_file(file_path, output_path):
    print(f"Processing: {file_path}...")
    df = pd.read_csv(file_path)
    
    # Áp dụng hàm tiền xử lý cho cột free_text
    df['clean_text'] = df['clean_text'].apply(preprocess_pipeline)
    
    # Lưu kết quả (giữ lại nhãn label_id)
    df[['clean_text', 'label_id']].to_csv(output_path, index=False, encoding='utf-8')
    print(f"Saved to: {output_path}")

# Danh sách các file cần xử lý
files = {
    "final_merge.csv":"final_merge_clean.csv"
}

for inp, out in files.items():
    try:
        process_file(inp, out)
    except FileNotFoundError:
        print(f"Không tìm thấy file {inp}, bỏ qua...")

print("Hoàn thành tiền xử lý!")

Processing: final_merge.csv...
Saved to: final_merge_clean.csv
Hoàn thành tiền xử lý!


In [13]:
import pandas as pd

df = pd.read_csv("./vihsd_groupdata_merge/final_merge_clean.csv")

value_count = df['label_id'].value_counts()
value_count_percentage = df['label_id'].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "Số lượng" : value_count,
    "Tỉ lệ": value_count_percentage
})

print(summary)

          Số lượng      Tỉ lệ
label_id                     
0            25797  39.835390
2            22491  34.730308
1            16471  25.434303
